# Practical No. 04 — Language Models (N-Gram Approach)

**Name:** Krishna Mungase  
**PRN:** 202301040020

---

An **N-Gram** is a contiguous sequence of N words from a text. An N-Gram language model predicts the next word based on the previous N−1 words.

- **Unigram (N=1)**: single words
- **Bigram (N=2)**: pair of consecutive words
- **Trigram (N=3)**: three consecutive words

**Steps:** Tokenize → Generate N-grams → Count frequencies → Compute probabilities → Generate sentences → Evaluate with perplexity.

## 1. Generate N-Grams

Split a string into tokens and form every contiguous N-word window.

In [ ]:
def generate_ngrams(text, n):
    tokens = text.split()
    ngrams = [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]
    return ngrams


text = "Geeks for Geeks Community"

unigrams = generate_ngrams(text, 1)
bigrams = generate_ngrams(text, 2)
trigrams = generate_ngrams(text, 3)

print("Unigrams:", unigrams)
print("Bigrams:", bigrams)
print("Trigrams:", trigrams)

Unigrams: [('Geeks',), ('for',), ('Geeks',), ('Community',)]
Bigrams: [('Geeks', 'for'), ('for', 'Geeks'), ('Geeks', 'Community')]
Trigrams: [('Geeks', 'for', 'Geeks'), ('for', 'Geeks', 'Community')]


In [ ]:
from collections import Counter

def count_ngrams(ngrams):
    return Counter(ngrams)

## 2. Count Frequencies

Use `Counter` to tally how often each n-gram appears.

In [3]:
bigram_counts = count_ngrams(bigrams)
unigram_counts = count_ngrams(unigrams)

print("Bigram Counts:", bigram_counts)
print("Unigram Counts:", unigram_counts)

Bigram Counts: Counter({('Geeks', 'for'): 1, ('for', 'Geeks'): 1, ('Geeks', 'Community'): 1})
Unigram Counts: Counter({('Geeks',): 2, ('for',): 1, ('Community',): 1})


In [4]:
def bigram_probabilities(bigram_counts, unigram_counts):
    probs = {}
    for (w1, w2), count in bigram_counts.items():
        probs[(w1, w2)] = count / unigram_counts[(w1,)]
    return probs


## 3. Bigram Probabilities

$$ P(w_2 \mid w_1) = \frac{\text{Count}(w_1, w_2)}{\text{Count}(w_1)} $$

In [5]:
bigram_probs = bigram_probabilities(bigram_counts, unigram_counts)
print("Bigram Probabilities:", bigram_probs)


Bigram Probabilities: {('Geeks', 'for'): 0.5, ('for', 'Geeks'): 1.0, ('Geeks', 'Community'): 0.5}


## 4. Worked Example — Corpus

Corpus:

```
I love NLP
I love AI
```

Apply the full pipeline: tokenize → unigrams/bigrams → counts → bigram probabilities.

In [ ]:
corpus = [
    "i love nlp",
    "i love ai",
]

all_unigrams = []
all_bigrams = []

for sentence in corpus:
    tokens = sentence.split()
    all_unigrams += generate_ngrams(sentence, 1)
    all_bigrams += generate_ngrams(sentence, 2)

unigram_counts = count_ngrams(all_unigrams)
bigram_counts = count_ngrams(all_bigrams)

print("Unigram Counts:", dict(unigram_counts))
print("Bigram Counts :", dict(bigram_counts))

In [ ]:
bigram_probs = bigram_probabilities(bigram_counts, unigram_counts)

for (w1, w2), p in bigram_probs.items():
    print(f"P({w2} | {w1}) = {p}")

## 5. Sentence Generation

Start with a seed word, then repeatedly pick the next word from the bigram probability distribution until we reach the desired length.

In [ ]:
import random

def generate_sentence(start_word, bigram_probs, length=5):
    sentence = [start_word]
    current = start_word

    for _ in range(length - 1):
        candidates = [(w2, p) for (w1, w2), p in bigram_probs.items() if w1 == current]
        if not candidates:
            break
        words, probs = zip(*candidates)
        current = random.choices(words, weights=probs, k=1)[0]
        sentence.append(current)

    return " ".join(sentence)


random.seed(0)
for _ in range(5):
    print(generate_sentence("i", bigram_probs, length=4))

## 6. Evaluation — Perplexity

Perplexity measures how "confused" a language model is when predicting a sequence. **Lower is better.**

$$ PP(W) = \left( \prod_{i=1}^{N} \frac{1}{P(w_i \mid w_{i-1})} \right)^{1/N} $$

We add a small smoothing constant so unseen bigrams don't produce a zero probability (which would make perplexity infinite).

In [ ]:
import math

def perplexity(sentence, bigram_probs, smoothing=1e-6):
    tokens = sentence.split()
    bigrams = [(tokens[i], tokens[i + 1]) for i in range(len(tokens) - 1)]

    log_prob_sum = 0.0
    for bg in bigrams:
        p = bigram_probs.get(bg, smoothing)
        log_prob_sum += math.log(p)

    n = len(bigrams)
    return math.exp(-log_prob_sum / n)


test_sentences = [
    "i love nlp",
    "i love ai",
    "i love python",
]

for s in test_sentences:
    print(f"Perplexity('{s}') = {perplexity(s, bigram_probs):.4f}")

## Conclusion

- The N-Gram model predicts the next word using only the previous N−1 words.
- Probabilities are estimated from relative frequencies on a training corpus.
- Sentences like `i love nlp` / `i love ai` sit in the training data and therefore yield low perplexity, while unseen continuations like `i love python` produce a much larger value, exposing the model's **data sparsity** limitation.

**Advantages:** simple, fast to train, easy to understand.  
**Limitations:** cannot capture long-range dependencies and struggles with unseen n-grams without smoothing.